# 06 - Treino do Modelo com MLflow

No notebook 05 eu preparei a `gold_ml_features`, com as colunas que o modelo vai usar (medias moveis, volatilidade, retornos passados) e **dois targets** pra comparar: `target_retorno_prox_dia` e `target_volatilidade_futura_5d`. Agora vem a parte de treinar de verdade e acompanhar isso com o **MLflow**.

**Por que dois modelos?** O primeiro testa o retorno do proximo dia. O segundo testa a volatilidade dos cinco pregões futuros, calculada sem compartilhar dias com a volatilidade historica usada como feature. A comparacao serve pra avaliar duas hipoteses diferentes, sem assumir antecipadamente que uma delas vai funcionar.

**Por que MLflow?** Cada modelo treinado vira um "run" registrado com parametros, metricas e o modelo salvo, sem precisar de setup extra no Databricks.

**Por que saio do Spark aqui?** A `gold_ml_features` tem pouco mais de 10 mil linhas nesta execucao, um volume pequeno para processamento em memoria. scikit-learn e XGBoost trabalham com pandas/numpy, entao faz mais sentido converter pra pandas do que manter a complexidade do Spark pra um dataset desse tamanho.

O plano deste notebook:
1. Ler a `gold_ml_features` e converter pra pandas
2. Separar treino e teste respeitando a ordem do tempo e uma faixa de seguranca de 5 dias
3. Treinar o modelo de **retorno** (primeira hipotese, esperado ser fraco)
4. Treinar o modelo de **volatilidade futura**
5. Comparar o XGBoost com um baseline simples
6. Avaliar MAE, RMSE e R2

In [0]:
%pip install mlflow xgboost scikit-learn

In [0]:
dbutils.library.restartPython()

## Passo 1 -- Lendo a feature store

O `restartPython()` limpa a memoria do notebook (por isso os pacotes novos entram em vigor), entao preciso ler os dados de novo aqui.

`.toPandas()` converte um DataFrame do Spark pra um DataFrame do pandas. So faco isso porque o dataset e pequeno -- se fossem milhoes de linhas, essa conversao ia estourar a memoria e eu precisaria treinar com Spark ML em vez disso.

In [0]:
df_pd = spark.table("b3_pipeline.gold_ml_features").toPandas()
df_pd = df_pd.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Total de linhas: {len(df_pd)}")
df_pd.head()

## Passo 2 -- Separando treino e teste (respeitando o tempo)

Com serie temporal nao da pra usar `train_test_split` embaralhando as linhas aleatoriamente -- o modelo acabaria treinando com dados de datas futuras e sendo testado com datas passadas, o que se chama **vazamento de dados (data leakage)**. O resultado pareceria otimo no teste, mas seria inutil na vida real.

O target de volatilidade olha 5 pregões pra frente. Por isso nao basta cortar exatamente numa data: as ultimas linhas do treino ainda poderiam usar como target dias que pertencem ao teste. Deixo uma faixa de seguranca (*purge gap*) com 5 datas entre os conjuntos. Assim, nenhum target do treino atravessa o inicio do teste.

In [0]:
# Pego todas as datas unicas em ordem e separo os 20% finais para teste
datas_unicas = sorted(df_pd["date"].unique())
indice_corte = int(len(datas_unicas) * 0.8)
data_inicio_teste = datas_unicas[indice_corte]

# Como o maior horizonte do target e 5 dias, retiro as 5 datas anteriores ao teste
horizonte_target = 5
data_inicio_purga = datas_unicas[indice_corte - horizonte_target]

treino = df_pd[df_pd["date"] < data_inicio_purga]
teste = df_pd[df_pd["date"] >= data_inicio_teste]

print(f"Treino: {len(treino)} linhas | Teste: {len(teste)} linhas")
print(f"Inicio da faixa de seguranca: {data_inicio_purga}")
print(f"Inicio do teste: {data_inicio_teste}")

## Passo 3 -- Escolhendo features (X)

As features de entrada sao as mesmas pros dois modelos -- o que muda e so o `y` (o target). Deixei de fora `ticker`, `setor`, `date` e `close` por enquanto: sao texto ou data, e um modelo de regressao simples como esse precisa de numeros.

In [0]:
features = [
    "daily_return_pct",
    "media_movel_5d", "media_movel_10d", "volatilidade_5d",
    "retorno_lag1", "retorno_lag3", "retorno_lag5"
]

X_treino = treino[features]
X_teste = teste[features]

## Passo 4 -- Modelo 1: prevendo o retorno do dia seguinte

Uso o **XGBoost**, que e um modelo de *gradient boosting*: monta varias arvores de decisao pequenas em sequencia, cada uma corrigindo o erro que as anteriores deixaram passar.

`with mlflow.start_run():` abre um "run" -- tudo que acontece dentro desse bloco (parametros, metricas, modelo) fica registrado junto. Duas metricas de avaliacao:
- **MAE**: erro medio absoluto, na mesma unidade do target
- **RMSE**: parecido com o MAE, mas penaliza mais os erros grandes
- **R2**: quanto o modelo explica da variacao do alvo (pode ser negativo se for pior que prever a media)

Aviso antecipado: esse aqui deve sair fraco (R2 perto de zero). E o esperado -- guarda esse numero pra comparar com o proximo modelo.

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_treino_retorno = treino["target_retorno_prox_dia"]
y_teste_retorno = teste["target_retorno_prox_dia"]

with mlflow.start_run(run_name="xgboost_retorno_prox_dia"):
    parametros = {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42
    }

    modelo_retorno = XGBRegressor(**parametros)
    modelo_retorno.fit(X_treino, y_treino_retorno)

    y_pred_retorno = modelo_retorno.predict(X_teste)
    mae_retorno = mean_absolute_error(y_teste_retorno, y_pred_retorno)
    rmse_retorno = np.sqrt(mean_squared_error(y_teste_retorno, y_pred_retorno))
    r2_retorno = r2_score(y_teste_retorno, y_pred_retorno)

    for nome, valor in parametros.items():
        mlflow.log_param(nome, valor)
    mlflow.log_metric("mae", mae_retorno)
    mlflow.log_metric("rmse", rmse_retorno)
    mlflow.log_metric("r2", r2_retorno)
    mlflow.xgboost.log_model(modelo_retorno, "modelo")

    print(f"[Retorno] MAE: {mae_retorno:.4f}")
    print(f"[Retorno] RMSE: {rmse_retorno:.4f}")
    print(f"[Retorno] R2: {r2_retorno:.4f}")

## Passo 5 -- Modelo 2: prevendo a volatilidade futura de 5 dias

Mesma estrutura de treino, mesmas features -- so troco o `y` pra `target_volatilidade_futura_5d`. Agora o target e calculado com os 5 pregões posteriores ao dia da feature, sem sobreposicao entre as duas janelas.

Tambem crio um **baseline de persistencia**, que simplesmente usa a volatilidade atual como previsao da volatilidade futura. O XGBoost so agrega valor se superar esse palpite simples. As metricas do modelo e do baseline ficam registradas no mesmo run do MLflow.

In [0]:
y_treino_vol = treino["target_volatilidade_futura_5d"]
y_teste_vol = teste["target_volatilidade_futura_5d"]

with mlflow.start_run(run_name="xgboost_volatilidade_futura_5d"):
    parametros = {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42
    }

    modelo_vol = XGBRegressor(**parametros)
    modelo_vol.fit(X_treino, y_treino_vol)

    y_pred_vol = modelo_vol.predict(X_teste)
    mae_vol = mean_absolute_error(y_teste_vol, y_pred_vol)
    rmse_vol = np.sqrt(mean_squared_error(y_teste_vol, y_pred_vol))
    r2_vol = r2_score(y_teste_vol, y_pred_vol)

    # Baseline: usa a volatilidade historica atual como previsao dos proximos 5 dias
    y_pred_baseline = X_teste["volatilidade_5d"]
    mae_baseline = mean_absolute_error(y_teste_vol, y_pred_baseline)
    rmse_baseline = np.sqrt(mean_squared_error(y_teste_vol, y_pred_baseline))
    r2_baseline = r2_score(y_teste_vol, y_pred_baseline)

    for nome, valor in parametros.items():
        mlflow.log_param(nome, valor)
    mlflow.log_metric("mae", mae_vol)
    mlflow.log_metric("rmse", rmse_vol)
    mlflow.log_metric("r2", r2_vol)
    mlflow.log_metric("baseline_mae", mae_baseline)
    mlflow.log_metric("baseline_rmse", rmse_baseline)
    mlflow.log_metric("baseline_r2", r2_baseline)
    mlflow.xgboost.log_model(modelo_vol, "modelo")

    print(f"[XGBoost volatilidade] MAE: {mae_vol:.4f}")
    print(f"[XGBoost volatilidade] RMSE: {rmse_vol:.4f}")
    print(f"[XGBoost volatilidade] R2: {r2_vol:.4f}")
    print(f"[Baseline persistencia] MAE: {mae_baseline:.4f}")
    print(f"[Baseline persistencia] RMSE: {rmse_baseline:.4f}")
    print(f"[Baseline persistencia] R2: {r2_baseline:.4f}")

## Passo 6 -- Comparando modelos e baseline

A tabela final mostra os dois experimentos e inclui o baseline de volatilidade. MAE e RMSE menores sao melhores; R2 maior e melhor. A comparacao decisiva e entre `xgboost_volatilidade_futura_5d` e `baseline_persistencia_volatilidade`, porque os dois tentam prever exatamente o mesmo target.

In [0]:
import pandas as pd

comparacao = pd.DataFrame({
    "modelo": [
        "xgboost_retorno_prox_dia",
        "xgboost_volatilidade_futura_5d",
        "baseline_persistencia_volatilidade"
    ],
    "target": [
        "retorno_prox_dia",
        "volatilidade_futura_5d",
        "volatilidade_futura_5d"
    ],
    "mae": [mae_retorno, mae_vol, mae_baseline],
    "rmse": [rmse_retorno, rmse_vol, rmse_baseline],
    "r2": [r2_retorno, r2_vol, r2_baseline]
})

display(comparacao)

## Onde ver os resultados

No menu lateral do Databricks, clique em **Experiments** -- os dois runs (`xgboost_retorno_prox_dia` e `xgboost_volatilidade_futura_5d`) vao estar la. O run de volatilidade guarda tambem as metricas do baseline.

O resultado precisa ser lido sem procurar apenas o maior R2. Primeiro verifico se o XGBoost supera o baseline no mesmo target. Se superar, o proximo passo e repetir essa comparacao em varias janelas temporais com validacao walk-forward.